In [1]:
%pip install tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import re
import copy
import math
import numpy as np
import scipy.signal as signal
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from tqdm.auto import tqdm


/home/durgesh/.conda/envs/tf210/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def bandpass_filter(signal_data, fs=100, lowcut=0.5, highcut=40, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = signal.butter(order, [low, high], btype='band')
    return signal.filtfilt(b, a, signal_data)

def notch_filter(signal_data, fs=100, freq=50.0, Q=30.0):
    nyq = 0.5 * fs
    w0 = freq / nyq
    b, a = signal.iirnotch(w0, Q)
    return signal.filtfilt(b, a, signal_data)

def preprocess_signal(sig, fs=100):
    sig = np.asarray(sig, dtype=np.float32)
    sig = np.nan_to_num(sig, nan=0.0, posinf=0.0, neginf=0.0)

    sig = bandpass_filter(sig, fs)
    sig = notch_filter(sig, fs)

    sig = np.nan_to_num(sig, nan=0.0, posinf=0.0, neginf=0.0)
    std = np.std(sig)
    if std < 1e-8:
        return np.zeros_like(sig, dtype=np.float32)

    sig = (sig - np.mean(sig)) / (std + 1e-8)
    return sig.astype(np.float32)


In [4]:
def segment_signal(signal_data, annotations, fs=100):
    segments = []
    labels = []
    segment_length = fs * 60

    for i, label in enumerate(annotations):
        start = i * segment_length
        end = start + segment_length

        if end <= len(signal_data):
            seg = signal_data[start:end]
            if not np.isfinite(seg).all():
                continue
            segments.append(seg)
            labels.append(1 if label == 'A' else 0)

    return np.array(segments, dtype=np.float32), np.array(labels, dtype=np.int64)


In [5]:
def _read_edf_channel(path, channel_index=0):
    """Read one channel from EDF/REC file without external deps."""
    with open(path, "rb") as f:
        _ = f.read(8)  # version
        _ = f.read(80)  # patient id
        _ = f.read(80)  # recording id
        _ = f.read(8)   # start date
        _ = f.read(8)   # start time
        header_bytes = int(f.read(8).decode("ascii", errors="ignore").strip() or 0)
        _ = f.read(44)  # reserved
        n_records = int(float((f.read(8).decode("ascii", errors="ignore").strip() or "0")))
        record_duration = float(f.read(8).decode("ascii", errors="ignore").strip() or 1.0)
        n_signals = int(f.read(4).decode("ascii", errors="ignore").strip() or 1)

        def _read_ascii_fields(width, n):
            raw = f.read(width * n)
            vals = [raw[i*width:(i+1)*width].decode("ascii", errors="ignore").strip() for i in range(n)]
            return vals

        labels = _read_ascii_fields(16, n_signals)
        _ = _read_ascii_fields(80, n_signals)  # transducer
        _ = _read_ascii_fields(8, n_signals)   # physical dimension
        phys_min = np.array([float(x or 0.0) for x in _read_ascii_fields(8, n_signals)], dtype=np.float64)
        phys_max = np.array([float(x or 1.0) for x in _read_ascii_fields(8, n_signals)], dtype=np.float64)
        dig_min = np.array([float(x or -32768) for x in _read_ascii_fields(8, n_signals)], dtype=np.float64)
        dig_max = np.array([float(x or 32767) for x in _read_ascii_fields(8, n_signals)], dtype=np.float64)
        _ = _read_ascii_fields(80, n_signals)  # prefilter
        samples_per_record = np.array([int(float(x or 0)) for x in _read_ascii_fields(8, n_signals)], dtype=np.int64)
        _ = _read_ascii_fields(32, n_signals)  # reserved per signal

        if channel_index >= n_signals:
            raise ValueError(f"channel_index={channel_index} out of range for {path}, n_signals={n_signals}")

        f.seek(header_bytes, os.SEEK_SET)
        total_spr = int(samples_per_record.sum())
        ch_offset = int(samples_per_record[:channel_index].sum())
        ch_spr = int(samples_per_record[channel_index])

        chunks = []
        for _rec in range(n_records):
            rec_data = np.fromfile(f, dtype="<i2", count=total_spr)
            if rec_data.size < total_spr:
                break
            ch_data = rec_data[ch_offset:ch_offset + ch_spr].astype(np.float64)
            chunks.append(ch_data)

        if not chunks:
            raise ValueError(f"No samples read from {path}")

        sig_digital = np.concatenate(chunks, axis=0)
        dmin, dmax = dig_min[channel_index], dig_max[channel_index]
        pmin, pmax = phys_min[channel_index], phys_max[channel_index]
        scale = (pmax - pmin) / max((dmax - dmin), 1e-8)
        sig_physical = (sig_digital - dmin) * scale + pmin
        fs = float(ch_spr) / max(record_duration, 1e-8)

        return sig_physical.astype(np.float32), fs, labels[channel_index] if labels else f"ch{channel_index}"


def _parse_respiratory_events(respevt_path):
    events = []
    with open(respevt_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line or not re.match(r"^\d{2}:\d{2}:\d{2}", line):
                continue
            parts = line.split()
            if len(parts) < 3:
                continue
            t = parts[0]
            evt_type = parts[1]
            duration = None
            for tok in parts[2:]:
                if tok.isdigit():
                    duration = int(tok)
                    break
            if duration is None or duration <= 0:
                continue
            hh, mm, ss = [int(x) for x in t.split(":")]
            start_sec = hh * 3600 + mm * 60 + ss
            events.append((start_sec, duration, evt_type))
    return events


def _label_timeline_from_events(num_seconds, events):
    labels = np.zeros(num_seconds, dtype=np.int64)
    for start_sec, duration, _evt_type in events:
        s = max(0, int(start_sec))
        e = min(num_seconds, int(start_sec + duration))
        if e > s:
            labels[s:e] = 1
    return labels


def _segment_ucddb_signal(signal_data, second_labels, fs=100, segment_seconds=60):
    segment_len = int(fs * segment_seconds)
    n_segments = len(signal_data) // segment_len
    if n_segments == 0:
        return np.empty((0, segment_len), dtype=np.float32), np.empty((0,), dtype=np.int64)

    X = []
    y = []
    for i in range(n_segments):
        a = i * segment_len
        b = a + segment_len
        seg = signal_data[a:b]
        if not np.isfinite(seg).all():
            continue
        sec_a = int(i * segment_seconds)
        sec_b = int((i + 1) * segment_seconds)
        lbl = 1 if np.any(second_labels[sec_a:sec_b] == 1) else 0
        X.append(seg)
        y.append(lbl)

    return np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.int64)


def load_ucddb_dataset(data_path, target_fs=100, channel_index=0):
    X_all = []
    y_all = []
    skipped = []

    records_file = os.path.join(data_path, "RECORDS")
    with open(records_file, "r", encoding="utf-8", errors="ignore") as f:
        rec_lines = [ln.strip() for ln in f if ln.strip()]

    record_ids = sorted(set(os.path.basename(x).replace("_lifecard.edf", "").replace(".rec", "") for x in rec_lines))

    for idx, rec_id in enumerate(record_ids, 1):
        edf_path = os.path.join(data_path, f"{rec_id}_lifecard.edf")
        rec_path = os.path.join(data_path, f"{rec_id}.rec")
        signal_path = edf_path if os.path.exists(edf_path) else rec_path
        respevt_path = os.path.join(data_path, f"{rec_id}_respevt.txt")

        try:
            if not os.path.exists(signal_path):
                raise FileNotFoundError(f"Signal file not found for {rec_id}")
            if not os.path.exists(respevt_path):
                raise FileNotFoundError(f"Resp event file not found for {rec_id}")

            sig, fs_raw, ch_name = _read_edf_channel(signal_path, channel_index=channel_index)
            sig = np.nan_to_num(sig, nan=0.0, posinf=0.0, neginf=0.0)

            if abs(fs_raw - target_fs) > 1e-3:
                sig = signal.resample_poly(sig, up=int(target_fs), down=max(1, int(round(fs_raw))))
                fs = target_fs
            else:
                fs = int(round(fs_raw))

            sig = preprocess_signal(sig, fs=fs)
            events = _parse_respiratory_events(respevt_path)
            n_secs = int(len(sig) // fs)
            sec_labels = _label_timeline_from_events(n_secs, events)
            segs, lbls = _segment_ucddb_signal(sig, sec_labels, fs=fs, segment_seconds=60)

            if len(segs) == 0:
                skipped.append((rec_id, "no valid segments"))
                continue

            X_all.append(segs)
            y_all.append(lbls)

            if idx % 5 == 0 or idx == len(record_ids):
                print(f"Loaded {idx}/{len(record_ids)} records (last: {rec_id}, ch={ch_name}, fs={fs_raw:.2f}->{fs})")

        except Exception as e:
            skipped.append((rec_id, str(e)))

    if not X_all:
        sample_err = skipped[0][1] if skipped else "unknown"
        raise ValueError(f"No valid UCDDB records were loaded. Example error: {sample_err}")

    X_all = np.concatenate(X_all, axis=0).astype(np.float32)
    y_all = np.concatenate(y_all, axis=0).astype(np.int64)

    finite_ratio = float(np.isfinite(X_all).mean())
    class_counts = np.bincount(y_all, minlength=2)
    print(f"Total segments: {len(X_all)} | finite_ratio={finite_ratio:.6f}")
    print(f"Class counts -> Normal(0): {class_counts[0]}, Apnea(1): {class_counts[1]}")
    if skipped:
        print(f"Skipped records: {len(skipped)}")

    return X_all, y_all


In [6]:
def oversample_minority(X, y, random_state=42):
    rng = np.random.default_rng(random_state)
    idx0 = np.where(y == 0)[0]
    idx1 = np.where(y == 1)[0]
    if len(idx0) == 0 or len(idx1) == 0:
        return X, y

    if len(idx0) > len(idx1):
        major, minor = idx0, idx1
    else:
        major, minor = idx1, idx0

    extra = rng.choice(minor, size=(len(major) - len(minor)), replace=True)
    all_idx = np.concatenate([major, minor, extra])
    rng.shuffle(all_idx)
    return X[all_idx], y[all_idx]


data_path = "dataset/UCDDB/physionet.org/files/ucddb/1.0.0"

X, y = load_ucddb_dataset(data_path, target_fs=100, channel_index=0)

# 8:1:1 split (train:val:test)
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=42
)

# Oversample only the training split
X_train, y_train = oversample_minority(X_train, y_train, random_state=42)

print(f"Train samples: {len(X_train)}, Val samples: {len(X_val)}, Test samples: {len(X_test)}")
train_counts = np.bincount(y_train, minlength=2)
val_counts = np.bincount(y_val, minlength=2)
test_counts = np.bincount(y_test, minlength=2)
print(f"Train class counts -> 0: {train_counts[0]}, 1: {train_counts[1]}")
print(f"Val class counts   -> 0: {val_counts[0]}, 1: {val_counts[1]}")
print(f"Test class counts  -> 0: {test_counts[0]}, 1: {test_counts[1]}")


Loaded 5/25 records (last: ucddb007, ch=chan 1, fs=128.00->100)
Loaded 10/25 records (last: ucddb012, ch=chan 1, fs=128.00->100)
Loaded 15/25 records (last: ucddb018, ch=chan 1, fs=128.00->100)
Loaded 20/25 records (last: ucddb023, ch=chan 1, fs=128.00->100)
Loaded 25/25 records (last: ucddb028, ch=chan 1, fs=128.00->100)
Total segments: 12206 | finite_ratio=1.000000
Class counts -> Normal(0): 9491, Apnea(1): 2715
Train samples: 15184, Val samples: 1221, Test samples: 1221
Train class counts -> 0: 7592, 1: 7592
Val class counts   -> 0: 949, 1: 272
Test class counts  -> 0: 950, 1: 271


In [7]:
class ApneaDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(ApneaDataset(X_train, y_train), batch_size=32, num_workers=4, shuffle=True, pin_memory=True)
val_loader = DataLoader(ApneaDataset(X_val, y_val), batch_size=32, num_workers=4, shuffle=False, pin_memory=True)
test_loader = DataLoader(ApneaDataset(X_test, y_test), batch_size=32, num_workers=4, shuffle=False, pin_memory=True)


In [8]:
class PeriodicSparseAttentionFECG(nn.Module):
    """Periodicity-aware sparse attention with fixed local windows around periodic anchors."""
    def __init__(self,
                 fs: int,
                 d_model: int,
                 nhead: int,
                 bpm_min: int = 40,
                 bpm_max: int = 90,
                 bpm_step: int = 10,
                 attention_window_samples: int = 45,
                 k_top_peaks: int = 32,
                 attention_dropout: float = 0.1,
                 scale: float = None,
                 output_attention: bool = False):
        super().__init__()
        self.fs = fs
        self.d_model = d_model
        self.nhead = nhead
        self.d_head = d_model // nhead
        self.attention_window_samples = attention_window_samples
        self.k_top_peaks = k_top_peaks
        self.dropout = nn.Dropout(attention_dropout)
        self.scale = scale
        self.output_attention = output_attention
        self.scale = scale or (1.0 / math.sqrt(float(self.d_head)))
        self.factor = factor = 5 

        bpms = list(range(bpm_min, bpm_max + 1, bpm_step))
        self._periods_samples = [(60.0 * fs) / float(bpm) for bpm in bpms]

    def _get_periodic_indices(self, L_K, device):
        idx_set = []
        for p in self._periods_samples:
            p_int = max(1, int(round(p)))
            if p_int >= L_K:
                continue
            idx_set.extend(range(0, L_K, p_int))

        if not idx_set:
            step = max(1, L_K // 8)
            idx_set = list(range(0, L_K, step))

        idx = torch.tensor(sorted(set(idx_set)), dtype=torch.long, device=device)
        if idx.numel() > self.k_top_peaks:
            pick = torch.linspace(0, idx.numel() - 1, steps=self.k_top_peaks, device=device).long()
            idx = idx[pick]
        return idx
    
    def _get_initial_context(self, V, L_Q):
        B, H, _, D = V.shape
        v_mean = V.mean(dim=-2)
        return v_mean.unsqueeze(-2).expand(B, H, L_Q, D).clone()

    def _update_context(self, context, V, scores, selected_index):
        B, H, L_V, D = V.shape
        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        context_selected = torch.matmul(attn, V)
        b_idx = torch.arange(B, device=context.device)[:, None, None]
        h_idx = torch.arange(H, device=context.device)[None, :, None]
        context[b_idx, h_idx, selected_index, :] = context_selected.type_as(context)
        return context, None  # Simplified, no attn_map


    # def forward(self, queries, keys, values):
    #     B, L_Q, H, D_head = queries.shape
    #     _, L_K, _, _ = keys.shape
    #     device = queries.device

    #     Q = queries.transpose(1, 2)
    #     K = keys.transpose(1, 2)
    #     V = values.transpose(1, 2)

    #     periodic_indices = self._get_periodic_indices(L_K, device)
        
    #     attn_mask = torch.full((L_Q, L_K), float('-inf'), device=device)
    #     for peak_idx in periodic_indices:
    #         start = max(0, peak_idx.item() - self.attention_window_samples)
    #         end = min(L_K, peak_idx.item() + self.attention_window_samples + 1)
    #         attn_mask[:, start:end] = 0.0
    #     attn_mask = attn_mask.unsqueeze(0).unsqueeze(0).expand(B, H, -1, -1)

    #     scale = self.scale or (1.0 / math.sqrt(D_head))
    #     scores = torch.matmul(Q, K.transpose(-2, -1)) * scale
    #     scores += attn_mask

    #     attn_weights = torch.softmax(scores, dim=-1)
    #     attn_weights = self.dropout(attn_weights)

    #     context = torch.matmul(attn_weights, V)
    #     context = context.transpose(1, 2).contiguous()

    #     attn_map = attn_weights if self.output_attention else None
    #     return context, attn_map
    
    def forward(self, queries, keys, values):
        B, L_Q, H, D_head = queries.shape
        _, L_K, _, _ = keys.shape
        device = queries.device
            
        Q = queries.transpose(1, 2)  # [B,H,LQ,D]
        K = keys.transpose(1, 2)     # [B,H,LK,D]
        V = values.transpose(1, 2)
            
            # COPY FROM FECG: Periodic + sparse extraction
        U_part = self.factor * math.ceil(math.log(max(1, L_K)))  # Add self.factor=5 to __init__
        periodic_idx_cpu = self._get_periodic_indices(L_K, device)
        sample_k = min(U_part, periodic_idx_cpu.numel())
        periodic_idx = periodic_idx_cpu[:sample_k].to(device)
            
        idx_expand = periodic_idx.view(1,1,1,sample_k).expand(B,H,L_Q,sample_k)
        K_sample = torch.gather(K.unsqueeze(2).expand(-1,-1,L_Q,-1,-1), 3, 
                                idx_expand.unsqueeze(-1).expand(-1,-1,-1,-1,D_head))
            
            # Sparse QK (O(n log n))
        S = torch.matmul(Q.unsqueeze(-2), K_sample.transpose(-2,-1)).squeeze(-2)
        M_metric = S.max(dim=-1)[0] - S.mean(dim=-1)
        u_eff = max(1, self.factor * math.ceil(math.log(max(1, L_Q))))
        _, topk_idx = M_metric.topk(u_eff, dim=-1, largest=True, sorted=False)
            
            # Final sparse refine
        b_idx = torch.arange(B, device=device)[:,None,None]
        h_idx = torch.arange(H, device=device)[None,:,None]
        Q_reduce = Q[b_idx, h_idx, topk_idx, :]
        scores_top = torch.matmul(Q_reduce, K.transpose(-2,-1)) * self.scale
            
            # COPY FROM FECG: Iterative context update (your _update_context)
        context = self._get_initial_context(V, L_Q)  # Add these two methods from FECG
        context, attn_map = self._update_context(context, V, scores_top, topk_idx)
            
        context = context.transpose(1, 2).contiguous()
        return context, attn_map



class SparseAttentionBlock(nn.Module):
    """Projection wrapper around PeriodicSparseAttentionFECG."""
    def __init__(self, in_channels, nhead, fs, bpm_range, dropout=0.1, attention_window_samples=45, k_top_peaks=32):
        super().__init__()
        if in_channels % nhead != 0:
            raise ValueError(f"in_channels ({in_channels}) must be divisible by nhead ({nhead}).")

        self.nhead = nhead
        self.d_model = in_channels
        self.d_head = self.d_model // nhead

        self.query_projection = nn.Conv1d(in_channels, self.d_model, kernel_size=1)
        self.key_projection = nn.Conv1d(in_channels, self.d_model, kernel_size=1)
        self.value_projection = nn.Conv1d(in_channels, self.d_model, kernel_size=1)

        self.attention = PeriodicSparseAttentionFECG(
            fs=fs,
            d_model=self.d_model,
            nhead=nhead,
            bpm_min=bpm_range[0],
            bpm_max=bpm_range[1],
            bpm_step=10,
            attention_window_samples=attention_window_samples,
            k_top_peaks=k_top_peaks,
            attention_dropout=dropout,
        )

        self.out_projection = nn.Conv1d(self.d_model, in_channels, kernel_size=1)

    def forward(self, x):
        B, C, T = x.shape
        H = self.nhead
        D_head = self.d_head

        q = self.query_projection(x).view(B, H, D_head, T).permute(0, 3, 1, 2)
        k = self.key_projection(x).view(B, H, D_head, T).permute(0, 3, 1, 2)
        v = self.value_projection(x).view(B, H, D_head, T).permute(0, 3, 1, 2)

        attn_output, _ = self.attention(q, k, v)
        attn_output = attn_output.permute(0, 2, 3, 1).reshape(B, C, T)
        return self.out_projection(attn_output)


class PCSA(nn.Module):
    def __init__(self, channels, fs=100, bpm_min=40, bpm_max=90, bpm_step=10, nhead=8, attention_window_samples=45, k_top_peaks=32):
        super().__init__()
        if channels % nhead != 0:
            raise ValueError(f"channels ({channels}) must be divisible by nhead ({nhead}).")
        self.block = SparseAttentionBlock(
            in_channels=channels,
            nhead=nhead,
            fs=fs,
            bpm_range=(bpm_min, bpm_max),
            dropout=0.1,
            attention_window_samples=attention_window_samples,
            k_top_peaks=k_top_peaks,
        )

    def forward(self, x):
        return self.block(x)



In [9]:
class Conv1DBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size, padding=kernel_size // 2),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class Conv2DBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class TwoConvBranch2D(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = Conv2DBlock(channels, channels)
        self.conv2 = Conv2DBlock(channels, channels)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        return x


class CustomApneaModel(nn.Module):
    def __init__(self):
        super().__init__()

        # Multiscale transformation (Conv1d-BN-ReLU x3)
        self.ms1 = Conv1DBlock(1, 32, 3)
        self.ms2 = Conv1DBlock(1, 32, 5)
        self.ms3 = Conv1DBlock(1, 32, 7)

        # 2D feature extraction stem: Conv2d -> Pool -> Conv2d -> Pool
        self.stem_conv1 = Conv2DBlock(1, 32)
        self.pool1 = nn.MaxPool2d(kernel_size=(1, 2), stride=(1, 2))
        self.stem_conv2 = Conv2DBlock(32, 64)
        self.pool2 = nn.MaxPool2d(kernel_size=(1, 2), stride=(1, 2))

        # Three parallel 2x Conv2d-BN-ReLU branches
        self.branch1 = TwoConvBranch2D(64)
        self.branch2 = TwoConvBranch2D(64)
        self.branch3 = TwoConvBranch2D(64)

        self.fuse = Conv2DBlock(64 * 3, 96)

        # Final PCSA block, then two linear layers
        self.pcsa = PCSA(96)
        self.fc1 = nn.Linear(96, 64)
        self.fc2 = nn.Linear(64, 2)

    def forward(self, x):
        # x: [B, 1, L]
        m1 = self.ms1(x)
        m2 = self.ms2(x)
        m3 = self.ms3(x)

        # Stack multiscale outputs as a 2D map: [B, 1, 3, T]
        ms = torch.stack([m1.mean(dim=1), m2.mean(dim=1), m3.mean(dim=1)], dim=1).unsqueeze(1)

        z = self.stem_conv1(ms)
        z = self.pool1(z)
        z = self.stem_conv2(z)
        z = self.pool2(z)

        b1 = self.branch1(z)
        b2 = self.branch2(z)
        b3 = self.branch3(z)

        z = torch.cat([b1, b2, b3], dim=1)
        z = self.fuse(z)

        # Convert 2D feature map to sequence and apply PCSA
        B, C, H, W = z.shape
        z = z.view(B, C, H * W)
        z = self.pcsa(z)

        z = z.mean(dim=-1)
        z = F.relu(self.fc1(z))
        z = self.fc2(z)

        return z


In [10]:
if torch.cuda.is_available():
    free_memories = []
    for i in range(torch.cuda.device_count()):
        free_bytes, total_bytes = torch.cuda.mem_get_info(i)
        free_memories.append((free_bytes, i, total_bytes))

    free_bytes, best_gpu, total_bytes = max(free_memories, key=lambda x: x[0])
    torch.cuda.set_device(best_gpu)
    device = torch.device(f"cuda:{best_gpu}")
    print(
        f"Using GPU {best_gpu}: {torch.cuda.get_device_name(best_gpu)} | "
        f"free {free_bytes / (1024**3):.2f} GB / total {total_bytes / (1024**3):.2f} GB"
    )
else:
    device = torch.device("cpu")
    print("Using CPU")

model = CustomApneaModel().to(device)

class_counts = np.bincount(y_train, minlength=2).astype(np.float32)
class_weights = class_counts.sum() / (2.0 * (class_counts + 1e-8))
criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=device))

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(f"Device: {device}")
print(f"Class weights used in loss: {class_weights}")


Using GPU 1: NVIDIA RTX A4000 | free 6.06 GB / total 15.60 GB
Device: cuda:1
Class weights used in loss: [1. 1.]


In [11]:
from sklearn.metrics import confusion_matrix, accuracy_score

def train(model, loader, epoch_idx, num_epochs):
    model.train()
    total_loss = 0.0
    valid_batches = 0
    skipped_batches = 0

    pbar = tqdm(
        loader,
        total=len(loader),
        desc=f"Epoch {epoch_idx}/{num_epochs} Train",
        unit='batch',
        leave=True,
        dynamic_ncols=True,
        mininterval=0.2,
    )

    for X_batch, y_batch in pbar:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        if not torch.isfinite(X_batch).all():
            skipped_batches += 1
            pbar.set_postfix(skipped=skipped_batches)
            continue

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        if not torch.isfinite(loss):
            skipped_batches += 1
            pbar.set_postfix(skipped=skipped_batches)
            continue

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        valid_batches += 1
        running_loss = total_loss / max(valid_batches, 1)
        pbar.set_postfix(loss=f"{running_loss:.4f}", valid=valid_batches, skipped=skipped_batches)

    mean_loss = total_loss / max(valid_batches, 1)
    return mean_loss, valid_batches, skipped_batches


def evaluate(model, loader, epoch_idx, num_epochs):
    model.eval()
    preds = []
    truths = []

    with torch.no_grad():
        pbar = tqdm(
            loader,
            total=len(loader),
            desc=f"Epoch {epoch_idx}/{num_epochs} Eval",
            unit='batch',
            leave=False,
            dynamic_ncols=True,
            mininterval=0.2,
        )
        for X_batch, y_batch in pbar:
            X_batch = X_batch.to(device)
            outputs = model(X_batch)
            predicted = torch.argmax(outputs, dim=1)

            preds.extend(predicted.cpu().numpy())
            truths.extend(y_batch.numpy())

    preds = np.array(preds)
    truths = np.array(truths)

    tn, fp, fn, tp = confusion_matrix(truths, preds, labels=[0, 1]).ravel()

    accuracy = accuracy_score(truths, preds)
    sensitivity = tp / (tp + fn + 1e-8)
    specificity = tn / (tn + fp + 1e-8)
    pos_pred_rate = (preds == 1).mean()

    return accuracy, sensitivity, specificity, pos_pred_rate


In [12]:
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True 

In [15]:
num_epochs = 100
patience = 15
min_delta = 1e-4

best_acc = -1.0
best_epoch = 0
best_state = None
no_improve = 0

for epoch in range(1, num_epochs + 1):
    print()
    print(f"Starting epoch {epoch}/{num_epochs}")
    loss, valid_batches, skipped_batches = train(model, train_loader, epoch, num_epochs)
    val_acc, val_sen, val_spec, val_pos_rate = evaluate(model, val_loader, epoch, num_epochs)

    if val_acc > best_acc + min_delta:
        best_acc = val_acc
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        no_improve = 0
    else:
        no_improve += 1

    print(f"Epoch {epoch}/{num_epochs}")
    print(f"Loss: {loss:.4f} | valid_batches: {valid_batches} | skipped_batches: {skipped_batches}")
    print(f"Val Accuracy: {val_acc:.4f}")
    print(f"Val Sensitivity: {val_sen:.4f}")
    print(f"Val Specificity: {val_spec:.4f}")
    print(f"Val Positive prediction rate: {val_pos_rate:.4f}")
    print(f"Best Val Accuracy so far: {best_acc:.4f} (epoch {best_epoch})")
    print("-" * 40)

    if no_improve >= patience:
        print(f"Early stopping at epoch {epoch} (no improvement for {patience} epochs).")
        break

if best_state is not None:
    model.load_state_dict(best_state)
    print(f"Loaded best model from epoch {best_epoch} with val accuracy {best_acc:.4f}.")



Starting epoch 1/100


Epoch 1/100 Train: 100%|██████████| 475/475 [00:50<00:00,  9.36batch/s, loss=0.5142, skipped=0, valid=475]


Epoch 1/100
Loss: 0.5142 | valid_batches: 475 | skipped_batches: 0
Val Accuracy: 0.7191
Val Sensitivity: 0.6581
Val Specificity: 0.7366
Val Positive prediction rate: 0.3514
Best Val Accuracy so far: 0.7191 (epoch 1)
----------------------------------------

Starting epoch 2/100


Epoch 2/100 Train: 100%|██████████| 475/475 [00:58<00:00,  8.11batch/s, loss=0.5076, skipped=0, valid=475]


Epoch 2/100
Loss: 0.5076 | valid_batches: 475 | skipped_batches: 0
Val Accuracy: 0.7518
Val Sensitivity: 0.7132
Val Specificity: 0.7629
Val Positive prediction rate: 0.3432
Best Val Accuracy so far: 0.7518 (epoch 2)
----------------------------------------

Starting epoch 3/100


...

Epoch 28/100
Loss: 0.4405 | valid_batches: 475 | skipped_batches: 0
Val Accuracy: 0.7265
Val Sensitivity: 0.7904
Val Specificity: 0.7081
Val Positive prediction rate: 0.4029
Best Val Accuracy so far: 0.7920 (epoch 15)
----------------------------------------

Starting epoch 29/100


Epoch 29/100 Train: 100%|██████████| 475/475 [00:56<00:00,  8.35batch/s, loss=0.4374, skipped=0, valid=475]


Epoch 29/100
Loss: 0.4374 | valid_batches: 475 | skipped_batches: 0
Val Accuracy: 0.7682
Val Sensitivity: 0.7059
Val Specificity: 0.7861
Val Positive prediction rate: 0.3235
Best Val Accuracy so far: 0.7920 (epoch 15)
----------------------------------------

Starting epoch 30/100


Epoch 30/100 Train: 100%|██████████| 475/475 [00:56<00:00,  8.39batch/s, loss=0.4358, skipped=0, valid=475]
                                                                     

Epoch 30/100
Loss: 0.4358 | valid_batches: 475 | skipped_batches: 0
Val Accuracy: 0.7191
Val Sensitivity: 0.7831
Val Specificity: 0.7007
Val Positive prediction rate: 0.4070
Best Val Accuracy so far: 0.7920 (epoch 15)
----------------------------------------
Early stopping at epoch 30 (no improvement for 15 epochs).
Loaded best model from epoch 15 with val accuracy 0.7920.


In [17]:
# Final evaluation on the test set
# Use epoch labels for progress-bar compatible evaluate() signature.
final_acc, final_sen, final_spec, final_pos_rate = evaluate(model, test_loader, epoch_idx='Final', num_epochs='Final')

print("\nFinal Test Metrics")
print(f"Accuracy   : {final_acc:.4f}")
print(f"Sensitivity: {final_sen:.4f}")
print(f"Specificity: {final_spec:.4f}")
print(f"Pos. Rate  : {final_pos_rate:.4f}")



Final Test Metrics
Accuracy   : 0.7887
Sensitivity: 0.6605
Specificity: 0.8253
Pos. Rate  : 0.2826
